# Medical Chatbot RAG - Colab

This notebook builds a FAISS vector index from a medical PDF and runs a RAG chatbot.

**Steps:**
1. Install dependencies
2. Upload your medical PDF
3. Create FAISS index (GPU-accelerated)
4. Test the chatbot interactively

## 1. Install Dependencies

In [ ]:
!pip install -q torch transformers sentence-transformers langchain langchain-community langchain-huggingface langchain-text-splitters faiss-cpu pypdf

## 2. Verify GPU

In [ ]:
import torch
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")
else:
    print("WARNING: No GPU detected. Go to Runtime > Change runtime type > GPU")

## 3. Upload Medical PDF

Upload your `medical book.pdf` file. If you already have it in Google Drive, mount Drive instead.

In [ ]:
from google.colab import files
import os

os.makedirs("data", exist_ok=True)
os.makedirs("models", exist_ok=True)

print("Upload your medical book PDF:")
uploaded = files.upload()

for filename in uploaded:
    os.rename(filename, f"data/medical book.pdf")
    print(f"Saved: data/medical book.pdf")

## 4. Create FAISS Index (GPU-Accelerated)

This loads the PDF, splits it into chunks, generates embeddings, and stores them in FAISS.

In [ ]:
import torch
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_community.document_loaders import PyPDFLoader

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {DEVICE}")

# Load PDF
loader = PyPDFLoader("data/medical book.pdf")
documents = loader.load()
print(f"Loaded {len(documents)} pages")

# Split into chunks
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=100,
    separators=["\n\n", "\n", " ", ""],
)
chunks = text_splitter.split_documents(documents)
print(f"Created {len(chunks)} chunks")

# Generate embeddings and build FAISS index
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    model_kwargs={"device": DEVICE},
)

vector_store = FAISS.from_documents(chunks, embeddings)
vector_store.save_local("models")
print("FAISS index saved to models/")

## 5. Build QA Chain and Test

This loads the FAISS index, loads the LLM, and creates the RAG chain.

In [ ]:
import torch
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings
from langchain.chains import RetrievalQA
from langchain_core.prompts import PromptTemplate
from langchain_huggingface.llms import HuggingFacePipeline
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer, pipeline

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE = torch.float16 if DEVICE == "cuda" else torch.float32
MODEL_NAME = "google/flan-t5-base"

# Load FAISS index
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    model_kwargs={"device": DEVICE},
)
vector_store = FAISS.load_local(
    "models", embeddings, allow_dangerous_deserialization=True
)
print("FAISS index loaded")

# Load LLM
print(f"Loading {MODEL_NAME}...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME, torch_dtype=DTYPE)
if DEVICE == "cuda":
    model = model.to(DEVICE)

pipe = pipeline(
    "text2text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=256,
    temperature=0.3,
    device=0 if DEVICE == "cuda" else -1,
)
llm = HuggingFacePipeline(pipeline=pipe)
print("LLM loaded")

# Build QA chain
PROMPT_TEMPLATE = """
Use the following pieces of context to answer the medical question at the end.
If you don't know the answer, just say that you don't know, don't try to make up an answer.
Keep the answer concise and relevant to the clinical context.

Context:
{context}

Question: {question}

Answer:
"""

PROMPT = PromptTemplate(template=PROMPT_TEMPLATE, input_variables=["context", "question"])

qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",
    retriever=vector_store.as_retriever(search_kwargs={"k": 4}),
    chain_type_kwargs={"prompt": PROMPT},
    return_source_documents=True,
)
print("QA chain ready!")

## 6. Interactive Chat

Ask medical questions and get answers from your document.

In [ ]:
def ask(question):
    result = qa_chain.invoke({"query": question})
    print(f"Q: {question}")
    print(f"A: {result['result']}")
    print(f"\nSources: {len(result['source_documents'])} chunks retrieved")
    print("-" * 60)
    return result

# Test with sample questions
ask("What are the symptoms of diabetes?")
ask("How is hypertension treated?")

## 7. Chat Loop (Optional)

Run this cell for an interactive Q&A loop. Type `quit` to exit.

In [ ]:
print("Medical Chatbot Ready! Type 'quit' to exit.\n")
while True:
    question = input("You: ")
    if question.lower() in ["quit", "exit", "q"]:
        print("Goodbye!")
        break
    result = qa_chain.invoke({"query": question})
    print(f"Bot: {result['result']}\n")